# §64 — KIM without VideoMAE, **Nexar host**: L5dfi + L5dficap vs L0dfi, beside K5dfirr / K0dfi (s2024)

Rule: `core/docs/final/KIM_RUN.md` **§64**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. PRELIMINARY (n = 1). **Decides nothing; K5f stays the reference.**

| row | = | input | here |
|---|---|---|---|
| **L5dfi** | L0dfi + K5dfi's four `kim.*` flags (KIM on the CLIP-only trunk) | plain `Nexar_s8_ncc` | train + read |
| **L5dficap** | L0dfi + `kim.enabled`, `kim.prototypes`, `kim.capacity_control` (branch + losses, **no differences**) | plain | train + read |
| **L0dfi** | §36 (LaGoVAD trunk on K5dfi's recipe) | plain | reused; Nexar val + DAD read here (re-trained as `L0dfirr` only if the start gate fails) |
| K5dfirr / K0dfi | §53 (VideoMAE host, KIM on / off, matched start) | A2 V2-S bakes | reused, nothing re-run |

**A** = L5 − L0 · **B** = K5 − K0 · **I** = A − B (does VideoMAE mask KIM?) · **C** = L5 − L5cap (do the differences carry it?). Deciding sets: Nexar val, DoTA-CAP-dev, DADA; DAD labelled only.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, deps, helpers, preflight | no | 3-5 min |
| 5 | tests — **raises if red** | no | 2 min |
| 6 | stage; DAD plain == bake check | no | 5-10 min |
| 7 | train L5dfi, L5dficap (1,749 steps each) + start gate (-> L0dfirr if needed) | **yes** | 2-3 × L0dfi's time |
| 8 | reads: Nexar val, DoTA-CAP-dev, DADA, DAD per L row | yes | 20-40 min per row |
| 9 | report -> **paste `REPORTS/kim_clip_host_report_nexar.md` to Claude** | no | 5-10 min |

A re-run skips what is on Drive. **Watch the log: past 2 s/step, stop and paste it.** Thesis / Thesis-V2 are only read.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Paths, rows, inputs

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, Nexar caches, A2 bakes, KNN, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, earlier rows and reads; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 64. KIM without VideoMAE' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §64 -- re-upload kat-vad/'
assert (REPO / 'core/tools/kim_clip_host_report.py').is_file(), 'no kim_clip_host_report -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'KIM_CLIP_HOST_ROWS'), 'constants has no §64 rows -- re-upload kat-vad/'

K5       = constants.KIM_MAIN_ARM                  # "K5": names run / eval sub-folders only
SEED     = constants.KIM_SEED                      # 2024
OFF      = constants.V2_OFF
ENC      = constants.V5_ENCODER
IGN      = constants.DVS_ANCHOR_MODE_IGNORE
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
STRIDE_B = constants.V2_E1_STRIDE_BC               # DoTA s1[::3] = protocol B rows
KIM_FLAGS = constants.KIM_ABLATION_FLAGS           # the four K5dfi KIM flags
CAP_FLAGS = constants.KIM_CLIP_HOST_CAP_FLAGS      # Kcap: enabled, prototypes, capacity_control
RR       = constants.KIM_ABLATION_RERUN_SUFFIX
ROOT     = constants.OUTPUT_ROOT / constants.KIM_CLIP_HOST_OUTPUT_DIRNAME   # L5 / L5cap (+ L0 re-run)
REPORTS  = ROOT / 'REPORTS'
V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'   # defs of every read
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME                 # plain CLIP: every L row's DADA input
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'          # plain CLIP: every L row's DoTA input

HOST     = constants.KIM_CLIP_HOST_NEXAR
NAMES    = constants.KIM_CLIP_HOST_ROWS[HOST]      # L5dfi, L5dficap, L0dfi, K5dfirr, K0dfi
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # nexar_test is never read here
DAD_NAME, DAD_STRIDE = constants.DAD_DATASET, constants.DAD_STRIDE
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}' / constants.KNN_CACHE_FILENAME
CLIP_S8  = V2_CACHE / 'clip' / 'Nexar_s8_ncc'                     # L0dfi's training input (plain CLIP)
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET    # nexar_val rows (plain = the K rows' CLIP columns)
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
DAD_PLAIN = constants.CLIP_CACHE_DIR / f'{DAD_NAME}_s{DAD_STRIDE}_ncc'
A2_DAD   = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DAD_NAME}_s{DAD_STRIDE}_nexar'   # K rows' DAD input (check only)
L0_SAVED = constants.OUTPUT_ROOT / constants.KIM_LAGOVAD_NEXAR_OUTPUT_DIRNAME / NAMES['L0']   # §36
K_ROOTS  = {k: constants.OUTPUT_ROOT / constants.KIM_ABLATION_OUTPUT_DIRNAME / NAMES[k] for k in ('K5', 'K0')}   # §53
LOCAL    = Path('/content/kim_clip_host_nexar')
LOCAL.mkdir(parents=True, exist_ok=True)

INPUTS = [('Nexar_window', WINDOW / constants.LABELS_TRAIN_FILENAME), ('KNN cache', KNN), ('Nexar census', CENSUS),
          ('Nexar plain CLIP s8', CLIP_S8), ('Nexar CLIP s1', NEXAR_X), ('DoTA s1', DOTA_S1), ('T2 meta', T2 / constants.META_FILENAME),
          ('T2 census', T2_CENSUS), ('T2 plain CLIP', T2_CLIP), ('DADA annotation', ANNOT), ('DoTA metadata', DOTA / 'metadata_val.json'),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME), ('DAD plain CLIP', DAD_PLAIN), ('DAD A2 bake', A2_DAD),
          ('L0dfi verified', L0_SAVED / K5 / f's{SEED}' / 'verified.json'),
          ('L0dfi DoTA-CAP-dev read (§36)', L0_SAVED / 'eval' / CAP_DEV / K5 / 'clip_scores.npz'),
          ('L0dfi DADA read (§36)', L0_SAVED / 'eval' / 'dada' / K5 / 'clip_scores.npz')]
for k, root in K_ROOTS.items():
    INPUTS += [(f'{root.name} verified', root / K5 / f's{SEED}' / 'verified.json'),
               (f'{root.name} nexar_val read', root / 'eval' / 'nexar' / root.name / 'nexar_readout.json'),
               (f'{root.name} DoTA read', root / 'eval' / 'dota' / K5 / 'clip_scores.npz'),
               (f'{root.name} DADA read', root / 'eval' / 'dada' / K5 / 'clip_scores.npz'),
               (f'{root.name} DAD read', root / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json')]
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | rows {NAMES} | new runs -> {ROOT}')

## 3. Dependencies

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- training and reads need one'
df -h /content | tail -1

## 4. Helpers + preflight (CPU)

In [ ]:
import torch  # noqa: E402
import yaml  # noqa: E402

from core.config import Config                         # noqa: E402
from core.data.v2_splits import load_split             # noqa: E402
from core.tools import k5_edge_report as edge          # noqa: E402
from core.tools import kim_ablation_report as ka       # noqa: E402
from core.tools.dada_whole_eval import test_sources    # noqa: E402
from core.tools.v5_stage1 import _flat                 # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    if todo:
        print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def cfg_of(root):
    return _flat(yaml.safe_load((root / K5 / f's{SEED}' / 'config.yaml').read_text()))


def run_dir(root):
    return root / K5 / f's{SEED}'


def last_epoch(rows, key):
    last = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    return sum(r.get(key, float('nan')) for r in last) / len(last)


def check_run(root, kind):
    """J10, §31 floor, CH9 config diff to the saved L0. ``kind`` in {'L5', 'L5cap', 'L0rr'}."""
    rd = run_dir(root)
    rows = metrics_rows(rd / 'metrics.jsonl')
    assert ckpt_step(rd / 'checkpoint_last.pt') == rows[-1]['global_step'] == STEPS, f'{root.name}: J10'
    got = cfg_of(root)
    if kind == 'L5':
        print(f'{root.name}: KIM flags vs {L0_SAVED.name}:', ka.check_kim_only(got, l0_cfg, root.name))
    elif kind == 'L5cap':
        diff = ka.config_diff(l0_cfg, got)
        assert diff == {k: (False, True) for k in CAP_FLAGS}, f'{root.name}: diff {diff} (§64 CH9)'
        print(f'{root.name}: Kcap flags vs {L0_SAVED.name}:', sorted(diff))
    else:
        diff = ka.config_diff(l0_cfg, got)
        assert not diff, f'{root.name}: differs from {L0_SAVED.name} by {diff} (a re-run allows nothing)'
    mul = last_epoch(rows, 'mul_mil')
    assert mul > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{root.name}: last-epoch mul_mil {mul:.4f} -- paste to Claude'
    keys = ('mil', 'mul_mil', 'kim_mil', 'kim_nkc')
    print('  last epoch: ' + ' | '.join(f'{k} {last_epoch(rows, k):.4f}' for k in keys))


def train_row(name, sets, kind):
    """Train one row into ROOT/<name> (resumable, atomic publish, J10, env recorded)."""
    root = ROOT / name
    rd, ckpt = run_dir(root), run_dir(root) / 'checkpoint_last.pt'
    if is_verified(rd, STEPS):
        print(f'{name} s{SEED}: verified on Drive -- not retrained')
        check_run(root, kind)
        return root
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / name
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': str(torch.version.cuda)}
    print(f'{name}: trains on {env}')
    (local / constants.KIM_RUN_ENV_FILENAME).write_text(json.dumps(env))
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in sets for x in ('--set', s)], '--data-dir', L_DATA,
         '--clip-dir', L_TRAIN_X, '--output-dir', local, '--knn-cache', L_KNN, *resume], log=LOCAL / f'train_{name}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{name}: J10 locally {last["global_step"]}'
    for f in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml', constants.KIM_RUN_ENV_FILENAME):
        publish(local / f, rd / f)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{name}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', rd / 'verified.json')
    print(f'{name}: trained in {(time.time() - t0) / 60:.0f} min | step-1 mil {metrics_rows(local / "metrics.jsonl")[0]["mil"]:.6f}')
    check_run(root, kind)
    return root


def start(ctrl_root, new_root):
    st = edge.start_check(run_dir(ctrl_root), run_dir(new_root))
    print(f"CH3 step 1: {new_root.name} {st['new']:.6f} vs {ctrl_root.name} {st['control']:.6f} -> "
          f"{'matched' if st['same_draws'] else 'DIFFERENT START'}")
    return st['same_draws']


def read_dota(root):
    """DoTA-CAP-dev, protocol B on plain DoTA_s1_ncc[::3] (every L row)."""
    out = root / 'eval' / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / root.name / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', run_dir(root) / 'checkpoint_last.pt',
             '--input-dir', L_DOTA_S1, '--s1-dir', L_DOTA_S1, '--metadata', DOTA / 'metadata_val.json',
             '--split-file', DOTA / 'val_split.txt', '--data-dir', L_LAB, '--split', CAP_DEV, '--stride', STRIDE_B,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{root.name}_dota.log')
        publish(local, out)
    r = json.loads((out / 'protocol_b_readout.json').read_text())
    assert r['clips'] == len(CAP_IDS) and r['global_steps'] == {f's{SEED}': STEPS} and r['arm']['motion'] == OFF, r
    print(f"{root.name} DoTA-CAP-dev: macro {r['macro']['mean']:.4f}")


def read_dada(root):
    """DADA whole on plain DADA2000_orig CLIP (every L row)."""
    out = root / 'eval' / 'dada' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / root.name / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', run_dir(root) / 'checkpoint_last.pt',
             '--t2-dir', L_T2, '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', L_T2CLIP,
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{root.name}_dada.log')
        publish(local, out)
    r = json.loads((out / 'dada_readout.json').read_text())
    assert r['videos'] == len(DADA_IDS) and r['global_steps'] == {f's{SEED}': STEPS} and r['arm']['motion'] == OFF, r
    print(f"{root.name} DADA whole: macro {r['macro']['mean']:.4f}")


# Preflight on small files only (C39)
l0_cfg = cfg_of(L0_SAVED)
assert not l0_cfg['kim.enabled'] and l0_cfg['v2.motion'] == OFF and l0_cfg['model.motion_dim'] == 0, 'L0_SAVED is not L0dfi'
assert (l0_cfg['dvs.theta'], l0_cfg['loss.dvs_anchor_mode'], l0_cfg['data.dataset']) == (0.7, IGN, DATASET), 'L0_SAVED is not L0dfi'
STEPS = json.loads((run_dir(L0_SAVED) / 'verified.json').read_text())['step']      # 1,749
EPOCHS = l0_cfg['train.num_epochs']                                                 # 33
assert is_verified(run_dir(L0_SAVED), STEPS) and metrics_rows(run_dir(L0_SAVED) / 'metrics.jsonl')[-1]['global_step'] == STEPS, 'L0dfi: J10'
for k, root in K_ROOTS.items():
    assert json.loads((run_dir(root) / 'verified.json').read_text())['step'] == STEPS, f'{root.name}: not {STEPS} steps'
st_k = edge.start_check(run_dir(K_ROOTS['K0']), run_dir(K_ROOTS['K5']))
assert st_k['same_draws'], f'K5dfirr / K0dfi no longer start alike: {st_k}'
TRAIN_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_TRAIN))
DADA_IDS, CAP_IDS = test_sources(T2), sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(DADA_IDS) == 370 and len(DAD_IDS) == constants.DAD_TEST_VIDEOS and len(TRAIN_IDS) == 900
for name, d, ids in (('Nexar plain s8', CLIP_S8, TRAIN_IDS), ('Nexar CLIP s1', NEXAR_X, NEXAR_IDS), ('DoTA s1', DOTA_S1, CAP_IDS),
                     ('T2 plain', T2_CLIP, DADA_IDS), ('DAD plain', DAD_PLAIN, DAD_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
print(f'preflight OK: L0dfi step {STEPS} ({EPOCHS} epochs) | K pair start matched ({st_k["new"]:.6f}) | '
      f'{NEXAR_SPLIT} {len(NEXAR_IDS)}, DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}, DAD {len(DAD_IDS)}')

## 5. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_kim_clip_host.py',
     'core/tests/test_kim_clip_host_report.py', 'core/tests/test_kim_ablation_report.py', 'core/tests/test_kim.py',
     'core/tests/test_dvs_dataset.py'])

## 6. Stage inputs; same CLIP frames on DAD (plain vs the K rows' bake)

In [ ]:
L_DATA    = stage(WINDOW, LOCAL / 'stage' / 'data')
L_TRAIN_X = stage(CLIP_S8, LOCAL / 'stage' / 'nexar_s8', TRAIN_IDS)
L_KNN     = stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name)
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)
L_T2CLIP  = stage(T2_CLIP, LOCAL / 'stage' / 'dada_clip', DADA_IDS)
L_NEXAR_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
L_DAD     = stage(DAD_PLAIN, LOCAL / 'stage' / 'dad_plain', DAD_IDS)
L_DADMETA = stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta')
L_LAB     = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)
# DAD is the one set §36 never checked: the plain rows must equal the bake's CLIP columns
done = LOCAL / 'checks' / 'dad.ok'
if not done.exists():
    run([sys.executable, '-m', 'core.tools.zs_lagovad_readout', 'check-inputs', '--plain', L_DAD,
         '--bake', stage(A2_DAD, LOCAL / 'stage' / 'dad_bake', DAD_IDS), '--ids-file', write_ids(LOCAL / 'ids' / 'dad.txt', DAD_IDS),
         '--plain-stride', 1])
    shutil.rmtree(LOCAL / 'stage' / 'dad_bake', ignore_errors=True)
    done.parent.mkdir(parents=True, exist_ok=True)
    done.write_text('ok')
print('staged; DAD plain == bake CLIP columns')

## 7. Train **L5dfi**, **L5dficap**; start gate (§64 CH3)

Each = L0dfi's exact command + its KIM flags. KIM is built on a private RNG with a zero-init return, so step-1 `mil` must equal L0dfi's. If either does not, L0dfi's exact command is re-trained here as **L0dfirr** and becomes the control (all its reads are redone).

In [ ]:
# L0dfi's exact command (§36 R1); L5 / L5cap add only their KIM flags (§64 CH1)
L0_SETS = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
           f'data.dataset={DATASET}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           'kip.enabled=false', f'loss.dvs_anchor_mode={IGN}', 'data.is_egocentric=false',
           f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}', 'dvs.theta=0.7', 'dvs.theta_ego=0.85',
           f'v2.motion={OFF}', 'model.motion_dim=0']
SPEC = {NAMES['L5']: ([*L0_SETS, *[f'{k}=true' for k in KIM_FLAGS]], 'L5'),
        NAMES['L5cap']: ([*L0_SETS, *[f'{k}=true' for k in CAP_FLAGS]], 'L5cap'),
        NAMES['L0'] + RR: ([*L0_SETS, 'kim.enabled=false'], 'L0rr')}
ROWS = {'K5': K_ROOTS['K5'], 'K0': K_ROOTS['K0']}
ROWS['L5'] = train_row(NAMES['L5'], *SPEC[NAMES['L5']])
ROWS['L5cap'] = train_row(NAMES['L5cap'], *SPEC[NAMES['L5cap']])
ROWS['L0'] = L0_SAVED
rerun = ROOT / (NAMES['L0'] + RR)
if not (start(L0_SAVED, ROWS['L5']) and start(L0_SAVED, ROWS['L5cap'])) or is_verified(run_dir(rerun), STEPS):
    print(f'CH3: a start differs from the saved {NAMES["L0"]} -> re-train its exact command ({rerun.name}); it becomes the control')
    ROWS['L0'] = train_row(rerun.name, *SPEC[rerun.name])
    for k in ('L5', 'L5cap'):
        if not start(ROWS['L0'], ROWS[k]):
            print(f'{k}: UNMATCHED START even against the re-run -- the report will not decide it; paste to Claude')
print('rows:', {k: v.name for k, v in ROWS.items()})

## 8. Reads of every L row: Nexar val (`--save-scores`), DoTA-CAP-dev (protocol B), DADA whole, DAD — plain CLIP inputs

In [ ]:
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON, VIDEO_STATS_JSON  # noqa: E402


def read_nexar(root):
    """nexar_val with --save-scores; plain rows: no --stats-dir, no --video-s1-dir."""
    out = root / 'eval' / 'nexar' / root.name
    done = all((out / f).exists() for f in (READOUT_JSON, ITEM_AUCS_JSON, VIDEO_STATS_JSON)) and \
        len(list((out / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
    if not done:
        local = LOCAL / 'eval' / root.name / 'nexar'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', run_dir(root) / 'checkpoint_last.pt',
             '--census', CENSUS, '--clip-s1-dir', L_NEXAR_X, '--data-dir', L_DEFS, '--split', NEXAR_SPLIT,
             '--save-scores', '--out-dir', local], log=LOCAL / f'{root.name}_nexar.log')
        publish(local, out)
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS) and r['global_steps'] == {f's{SEED}': STEPS}, r['global_steps']
    print(f"{root.name} nexar_val: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | whole macro {r['whole']['macro']['mean']:.4f}")


def read_dad(root):
    out = root / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}'
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / root.name / 'dad'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', run_dir(root) / 'checkpoint_last.pt', '--set', f'data.dataset={DAD_NAME}',
             '--data-dir', L_DADMETA, '--clip-dir', L_DAD, '--score-norm', 'auto', '--save-scores', '--output-dir', local],
            log=LOCAL / f'{root.name}_dad.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(DAD_IDS), r
    print(f"{root.name} DAD: macro {r['auc_macro']:.4f} | micro {r['auc']:.4f}")


# the saved L0dfi: its §36 DoTA-CAP-dev read is copied to the layout every row uses (bit-identical)
if ROWS['L0'] == L0_SAVED:
    alias = L0_SAVED / 'eval' / 'dota' / K5
    if not (alias / 'clip_aucs.json').exists():
        local = LOCAL / 'alias_dota'
        shutil.rmtree(local, ignore_errors=True)
        shutil.copytree(L0_SAVED / 'eval' / CAP_DEV / K5, local)
        publish(local, alias)
for key in ('L0', 'L5', 'L5cap'):
    root = ROWS[key]
    read_nexar(root)
    read_dota(root)
    read_dada(root)
    read_dad(root)
    flush_drive()

## 9. Report (§64 CH5–CH9) -> **paste `REPORTS/kim_clip_host_report_nexar.md` to Claude**

In [ ]:
local = LOCAL / 'report'
shutil.rmtree(local, ignore_errors=True)
local.mkdir(parents=True)
args = [x for k, root in ROWS.items() for x in ('--row', k, root.name, root)]
run([sys.executable, '-m', 'core.tools.kim_clip_host_report', '--host', HOST, '--seed', SEED, *args, '--out-dir', local])
publish(local, REPORTS)
stem = f'{constants.KIM_CLIP_HOST_REPORT_STEM}_{HOST}'
print((REPORTS / f'{stem}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- paste REPORTS/{stem}.md to Claude')